# Merge Smart-Intersection Data

In [4]:
import pandas as pd
import os
import glob

folder_path = "../data/raw/smart-intersection"
file_list = glob.glob(os.path.join(folder_path, "*.xlsx"))

df_list = []
reference_columns = None
schema_mismatch_files = [] # 스키마가 다른 파일들을 추적하기 위한 리스트

for file in file_list:
    try:
        temp_df = pd.read_excel(file)
        filename = os.path.basename(file)
        
        # 1. 스키마(컬럼 구조) 일치 여부 확인
        current_columns = list(temp_df.columns)
        
        # 첫 번째 파일을 기준으로 설정
        if reference_columns is None:
            reference_columns = current_columns
        # 이후 파일들의 컬럼이 기준과 다르면 리스트에 저장
        elif current_columns != reference_columns:
            schema_mismatch_files.append((filename, current_columns))
            
        temp_df['source_file'] = filename
        df_list.append(temp_df)
        
    except Exception as e:
        print(f"데이터 로드 실패 ({file}): {e}")

if df_list:
    # 2. 데이터 병합
    smart_intersection = pd.concat(df_list, ignore_index=True)
    
    print("=== 1. 스키마(컬럼 구조) 동일성 검사 ===")
    if not schema_mismatch_files:
        print("✅ 32개 모든 파일의 스키마가 동일합니다.")
    else:
        print(f"⚠️ 스키마가 다른 파일이 {len(schema_mismatch_files)}개 발견되었습니다!")
        for f_name, cols in schema_mismatch_files:
            print(f" - {f_name} 컬럼: {cols}")
            
    print("\n=== 2. 헤더(컬럼명) 중복 삽입 검사 ===")
    # pd.read_excel은 기본적으로 첫 행을 컬럼명으로 인식하므로 
    # 병합 과정에서 헤더가 데이터로 들어갈 확률은 낮지만, 
    # 원본 엑셀 파일 중간에 헤더가 반복되어 있거나 메타데이터가 섞여 있을 수 있음.
    
    # 임의로 첫 번째 컬럼을 기준으로, 데이터 안에 첫 번째 컬럼명과 같은 문자열이 있는지 확인
    col_to_check = reference_columns[0]
    header_rows_idx = smart_intersection[smart_intersection[col_to_check] == col_to_check].index
    
    if len(header_rows_idx) > 0:
        print(f"⚠️ 데이터 내부에 헤더가 섞여 들어간 행이 {len(header_rows_idx)}개 발견되었습니다.")
        # 헤더 행 삭제 처리
        smart_intersection = smart_intersection.drop(header_rows_idx)
        print("  -> 해당 헤더 행들을 삭제 조치했습니다.")
    else:
        print("✅ 데이터 내부에 헤더가 섞여 들어가지 않고 깔끔하게 병합되었습니다.")

    print(f"\n=== 최종 데이터 크기 ===")
    print(f"Shape: {smart_intersection.shape}")
    print("\n최종 컬럼 목록:", smart_intersection.columns.tolist())
    display(smart_intersection.head())

else:
    print("불러올 데이터가 없습니다. 폴더 경로를 확인하세요.")

=== 1. 스키마(컬럼 구조) 동일성 검사 ===
✅ 32개 모든 파일의 스키마가 동일합니다.

=== 2. 헤더(컬럼명) 중복 삽입 검사 ===
✅ 데이터 내부에 헤더가 섞여 들어가지 않고 깔끔하게 병합되었습니다.

=== 최종 데이터 크기 ===
Shape: (220449, 29)

최종 컬럼 목록: ['일자', '교차로명', '접근로명', '합계', '00시', '01시', '02시', '03시', '04시', '05시', '06시', '07시', '08시', '09시', '10시', '11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시', '19시', '20시', '21시', '22시', '23시', 'source_file']


,일자,교차로명,접근로명,합계,00시,01시,02시,03시,04시,05시,...,15시,16시,17시,18시,19시,20시,21시,22시,23시,source_file
0,2024-05-01,2공단3,2공단3_북,11723,157,87,75,75,74,157,...,903,293,129,795,614,542,472,270,182,스마트교차로_통계_202405.xlsx
1,2024-05-01,2공단3,2공단3_남,11834,140,91,78,76,75,300,...,817,193,126,606,465,402,343,246,158,스마트교차로_통계_202405.xlsx
2,2024-05-01,2공단3,2공단3_서,3748,23,26,18,8,23,98,...,314,107,58,423,212,84,64,64,33,스마트교차로_통계_202405.xlsx
3,2024-05-02,2공단3,2공단3_남,16182,94,56,56,70,93,470,...,861,768,695,629,589,494,463,274,148,스마트교차로_통계_202405.xlsx
4,2024-05-02,2공단3,2공단3_서,6733,14,9,14,7,25,105,...,407,613,859,625,575,324,203,100,46,스마트교차로_통계_202405.xlsx


# 교차로별 일평균 교통량, 피크 교통량, 야간비율 산출

In [8]:
# --- 1. 일평균 교통량 계산 ---
# 교차로명, 일자별로 모든 접근로의 '합계'를 더함 (하루 총 교통량)
daily_vol = smart_intersection.groupby(['교차로명', '일자'])['합계'].sum().reset_index()

# 교차로명별로 전체 기간의 일평균 산출
avg_daily_vol = daily_vol.groupby('교차로명')['합계'].mean().reset_index(name='일평균_교통량')


# --- 2. 피크(Peak) 교통량 계산 ---
# 00시부터 23시까지의 시간대 컬럼명 리스트 생성
time_cols = [f'{str(i).zfill(2)}시' for i in range(24)]

# 교차로명, 일자별로 모든 접근로의 시간대별 교통량을 합산
hourly_vol = smart_intersection.groupby(['교차로명', '일자'])[time_cols].sum().reset_index()

# 교차로별로 각 시간대의 평균을 계산
hourly_avg = hourly_vol.groupby('교차로명')[time_cols].mean()

# 각 교차로에서 평균 교통량이 가장 높은 시간대의 값을 피크 교통량으로 추출
peak_vol = hourly_avg.max(axis=1).reset_index(name='피크_교통량')


# --- 3. 야간비율 계산 ---
# 야간 시간대 지정 (22시부터 익일 05시 전까지)
night_cols = ['22시', '23시', '00시', '01시', '02시', '03시', '04시']

# 원본 데이터에 각 행의 야간 시간대 교통량 합계 컬럼 추가
smart_intersection['야간합계'] = smart_intersection[night_cols].sum(axis=1)

# 교차로명별 전체 합계(24시간) 총합 산출
total_vol = smart_intersection.groupby('교차로명')['합계'].sum().reset_index()

# 교차로명별 야간합계 총합 산출
night_total_vol = smart_intersection.groupby('교차로명')['야간합계'].sum().reset_index()

# 병합하여 야간 비율 계산 (야간합계 / 전체합계)
night_ratio = pd.merge(total_vol, night_total_vol, on='교차로명')
night_ratio['야간_비율'] = night_ratio['야간합계'] / night_ratio['합계']
night_ratio = night_ratio[['교차로명', '야간_비율']]


# --- 4. 최종 데이터프레임 병합 ---
# 교차로명을 기준으로 3가지 지표를 하나로 합침
x_traffic = avg_daily_vol.merge(peak_vol, on='교차로명', how='left').merge(night_ratio, on='교차로명', how='left')

print("=== 집계 완료 ===")
print(f"교차로 개수: {len(x_traffic)}개 (목표: 72개)")
print(f"Shape: {x_traffic.shape}")
print(f"컬럼 목록: {x_traffic.columns.tolist()}")
display(x_traffic)

=== 집계 완료 ===
교차로 개수: 72개 (목표: 72개)
Shape: (72, 4)
컬럼 목록: ['교차로명', '일평균_교통량', '피크_교통량', '야간_비율']


,교차로명,일평균_교통량,피크_교통량,야간_비율
0,2공단3,33142.812950,2334.513875,0.056353
1,계룡APT3,34079.099692,2499.115108,0.067323
2,고재4,51469.181912,3659.837616,0.062489
3,공구단지4,40874.933196,3015.902364,0.059378
4,공단4,37968.327852,2641.913669,0.057347
...,...,...,...,...
67,충무로4,37388.386434,2749.747174,0.059421
68,터미널4,40398.125385,2667.154162,0.091852
69,하신3,60369.546763,4408.791367,0.062413
70,한들사거리,60438.720412,4308.957118,0.082646


# 교차로 형태 추출

In [9]:
import numpy as np
import pandas as pd

# 1. 교차로 형태 추출 함수 
def get_shape_safe(name):
    # 확실한 명칭 (명시적)
    if '삼거리' in name: return 3, '확실'
    if '사거리' in name: return 4, '확실'
    if '오거리' in name: return 5, '확실'
    if '육거리' in name: return 6, '확실'
    
    # 숫자로 끝나는 경우 (검증 필요)
    if name[-1] in ['3', '4', '5', '6']:
        # 숫자를 정수로 변환하여 반환하되, 플래그를 달아둠
        return int(name[-1]), '확인필요'
        
    # ③ 알 수 없는 경우
    return np.nan, '수동입력'

# 2. 함수 적용하여 두 개의 컬럼(형태, 신뢰도플래그) 생성
# x_traffic 데이터프레임이 있다고 가정
x_traffic[['교차로형태', '신뢰도']] = x_traffic['교차로명'].apply(
    lambda x: pd.Series(get_shape_safe(x))
)

# 3. 상태별 개수 확인
print("=== 교차로 형태 코딩 결과 ===")
print(x_traffic['신뢰도'].value_counts())

# 4. '확인필요' 또는 '수동입력' 리스트만 뽑아서 보기
check_list = x_traffic[x_traffic['신뢰도'].isin(['확인필요', '수동입력'])]

print(f"\n⚠️ 검증이 필요한 교차로: {len(check_list)}개")
if len(check_list) > 0:
    display(check_list[['교차로명', '교차로형태', '신뢰도']])

=== 교차로 형태 코딩 결과 ===
신뢰도
확인필요    61
확실      10
수동입력     1
Name: count, dtype: int64

⚠️ 검증이 필요한 교차로: 62개


,교차로명,교차로형태,신뢰도
0,2공단3,3.0,확인필요
1,계룡APT3,3.0,확인필요
2,고재4,4.0,확인필요
3,공구단지4,4.0,확인필요
4,공단4,4.0,확인필요
...,...,...,...
65,청삼4,4.0,확인필요
66,청솔4,4.0,확인필요
67,충무로4,4.0,확인필요
68,터미널4,4.0,확인필요


In [11]:
import pandas as pd
import numpy as np

# 1. 접근로명 개수로 실제 교차로 형태(지 수) 계산
intersection_shape = (
    smart_intersection.groupby('교차로명')['접근로명']
    .nunique()
    .reset_index(name='접근로개수')
)

# 2. 교차로명에서 명칭상 형태 추출
def extract_shape_from_name(name):
    if '삼거리' in name: return 3
    if '사거리' in name: return 4
    if '오거리' in name: return 5
    if '육거리' in name: return 6
    # 끝자리가 숫자인 경우 해당 숫자 반환
    if name[-1].isdigit():
        return int(name[-1])
    return np.nan

intersection_shape['명칭추론_형태'] = intersection_shape['교차로명'].apply(extract_shape_from_name)

# 3. 데이터 일치 여부 검증 (접근로 개수 == 명칭 추론 형태)
intersection_shape['검증_일치여부'] = intersection_shape['접근로개수'] == intersection_shape['명칭추론_형태']

print("=== 1. 교차로 형태 크로스체크 결과 ===")
print(intersection_shape['검증_일치여부'].value_counts())

# 불일치 또는 결측치(NaN) 확인
mismatch_cases = intersection_shape[~intersection_shape['검증_일치여부']]

if len(mismatch_cases) > 0:
    print(f"\n⚠️ 불일치 또는 확인이 필요한 교차로: {len(mismatch_cases)}개")
    display(mismatch_cases)
    # 지피티가 말한 '수집 장비가 여러 개 잡혀 있는 경우' 등 예외 상황일 수 있음
else:
    print("\n✅ 모든 교차로의 접근로 개수와 명칭 규칙이 완벽하게 일치합니다!")

# 4. 최종 교차로 형태 확정 ('접근로개수'를 최우선 기준으로 사용)
intersection_shape['최종_교차로형태'] = intersection_shape['접근로개수']

# 5. 기존에 만들어둔 x_traffic에 병합
# (x_traffic은 일평균, 피크, 야간비율이 계산되어 있는 72행 데이터프레임)
final_x_traffic = x_traffic.merge(
    intersection_shape[['교차로명', '최종_교차로형태']], 
    on='교차로명', 
    how='left'
)

# 6. 컬럼명 정리 및 최종 출력
final_x_traffic = final_x_traffic.rename(columns={'최종_교차로형태': '교차로형태'})

print("\n=== 2. 최종 병합 데이터 (x_traffic_shape) 미리보기 ===")
display(final_x_traffic.head())

=== 1. 교차로 형태 크로스체크 결과 ===
검증_일치여부
True     52
False    20
Name: count, dtype: int64

⚠️ 불일치 또는 확인이 필요한 교차로: 20개


,교차로명,접근로개수,명칭추론_형태,검증_일치여부
2,고재4,2,4.0,False
7,굴울(변전소)4,2,4.0,False
8,남부5,4,5.0,False
10,동부4,2,4.0,False
29,삼성SDI3,2,3.0,False
31,삼용4,2,4.0,False
32,상수도민원센터3,2,3.0,False
37,선경APT4,2,4.0,False
39,성성고가교3,1,3.0,False
43,수자인입구4,2,4.0,False



=== 2. 최종 병합 데이터 (x_traffic_shape) 미리보기 ===


,교차로명,일평균_교통량,피크_교통량,야간_비율,교차로형태,신뢰도,교차로형태
0,2공단3,33142.812950,2334.513875,0.056353,3.0,확인필요,3
1,계룡APT3,34079.099692,2499.115108,0.067323,3.0,확인필요,3
2,고재4,51469.181912,3659.837616,0.062489,4.0,확인필요,2
3,공구단지4,40874.933196,3015.902364,0.059378,4.0,확인필요,4
4,공단4,37968.327852,2641.913669,0.057347,4.0,확인필요,4


In [14]:
import numpy as np

# 1. '명칭추론_형태'를 기본값으로 사용하고, 결측치인 경우에만 '접근로개수'로 채움
intersection_shape['최종_교차로형태'] = intersection_shape['명칭추론_형태'].fillna(intersection_shape['접근로개수'])

# 2. x_traffic 원본 데이터에서 잘못 병합된 '교차로형태' 컬럼이 있다면 삭제
if '교차로형태' in x_traffic.columns:
    x_traffic = x_traffic.drop(columns=['교차로형태', '신뢰도'], errors='ignore')

# 3. 다시 병합
final_x_traffic = x_traffic.merge(
    intersection_shape[['교차로명', '최종_교차로형태']], 
    on='교차로명', 
    how='left'
)

# 4. 컬럼명 깔끔하게 정리
final_x_traffic = final_x_traffic.rename(columns={'최종_교차로형태': '교차로형태'})

print("=== 최종 병합 데이터 ===")
display(final_x_traffic.head())

# 5. 최종 추출
final_x_traffic.to_csv("../data/processed/x_traffic_shape.csv", index=False, encoding='utf-8-sig')
print("\n🎉 'x_traffic_shape.csv' 추출 완료!")

=== 최종 병합 데이터 ===


,교차로명,일평균_교통량,피크_교통량,야간_비율,교차로형태
0,2공단3,33142.812950,2334.513875,0.056353,3.0
1,계룡APT3,34079.099692,2499.115108,0.067323,3.0
2,고재4,51469.181912,3659.837616,0.062489,4.0
3,공구단지4,40874.933196,3015.902364,0.059378,4.0
4,공단4,37968.327852,2641.913669,0.057347,4.0



🎉 'x_traffic_shape.csv' 추출 완료!


In [13]:
import numpy as np

# 1. 도출 근거를 기록하는 새로운 컬럼 생성
# 명칭추론_형태가 결측치(NaN)가 아니면 1순위, 결측치면 2순위로 분류
intersection_shape['도출_근거'] = np.where(
    intersection_shape['명칭추론_형태'].notna(), 
    '1순위 (명칭 기반)', 
    '2순위 (접근로 개수 기반)'
)

# 2. 순위별 개수 집계
reason_counts = intersection_shape['도출_근거'].value_counts()

print("=== 교차로 형태 도출 근거별 개수 ===")
print(reason_counts)

# (선택) 2순위로 도출된 교차로 목록만 따로 확인해보기
second_priority_list = intersection_shape[intersection_shape['도출_근거'] == '2순위 (접근로 개수 기반)']
print(f"\n⚠️ 2순위(접근로 개수)로 도출된 교차로 목록 ({len(second_priority_list)}개):")
display(second_priority_list[['교차로명', '접근로개수', '최종_교차로형태']])

=== 교차로 형태 도출 근거별 개수 ===
도출_근거
1순위 (명칭 기반)        71
2순위 (접근로 개수 기반)     1
Name: count, dtype: int64

⚠️ 2순위(접근로 개수)로 도출된 교차로 목록 (1개):


,교차로명,접근로개수,최종_교차로형태
59,저리교차로,4,4.0
